In [0]:
%pip install arelle-release

In [0]:
from arelle import Cntlr, ModelManager
from pyspark.sql.functions import col, lit
from arelle.XbrlConst import parentChild
import zipfile
from pyspark.sql.functions import col

In [0]:
schema = 'finance_staging'
table_name = 'dim_taxonomy_staging'

In [0]:
dbutils.widgets.text("max_year", "", "Latest GAAP Version Year")
max_gaap_year_to_process = dbutils.widgets.get("max_year")

dbutils.widgets.text("min_year", "", "Earliest GAAP Version Year")
min_gaap_year_to_process = dbutils.widgets.get("min_year")

dbutils.widgets.text("target_catalog", "", "Target Catalog")
target_catalog = dbutils.widgets.get("target_catalog")

In [0]:
final_df = None

gaap_year_to_process_list = []
for year in range(int(min_gaap_year_to_process),int(max_gaap_year_to_process)+1):
    gaap_year_to_process_list.append(year)

for gaap_year_to_process in gaap_year_to_process_list:
    gaap_version_processed = f'us-gaap/{gaap_year_to_process}'
    output_path = f"/Volumes/operations/finance_staging/taxonomy/us-gaap-{gaap_year_to_process}.zip"


    with zipfile.ZipFile(output_path, 'r') as zip_ref:
        zip_ref.extractall("/Volumes/operations/finance_staging/taxonomy/")

    cntlr = Cntlr.Cntlr()
    model_manager = ModelManager.initialize(cntlr)


    if gaap_year_to_process < 2022:
        gaap_year_to_process_zip_file = str(gaap_year_to_process)+"-01-31"

        taxonomy_path = f"/Volumes/operations/finance_staging/taxonomy/us-gaap-{gaap_year_to_process_zip_file}/entire/us-gaap-entryPoint-std-{gaap_year_to_process_zip_file}.xsd"
        
    elif gaap_year_to_process >= 2022:
        gaap_year_to_process_zip_file = gaap_year_to_process

        taxonomy_path = f"/Volumes/operations/finance_staging/taxonomy/us-gaap-{gaap_year_to_process}/entire/us-gaap-entryPoint-std-{gaap_year_to_process}.xsd"

    else:
        pass

    model_xbrl = model_manager.load(taxonomy_path)

    labels = []

    for concept in model_xbrl.qnameConcepts.values():
        label = concept.label()
        
        if label:
            labels.append({
                "concept_qname": str(concept.qname),
                "label_text": label
            })
    
    labels_df = spark.createDataFrame(labels)

    presentation = []

    rel_set = model_xbrl.relationshipSet(parentChild)

    for rel in rel_set.modelRelationships:
        presentation.append({
            "parent": str(rel.fromModelObject.qname),
            "child": str(rel.toModelObject.qname),
            "order": rel.order,
            "linkrole": rel.linkrole
        })
    
    presentation_df = spark.createDataFrame(presentation)


    balance_sheet_df = presentation_df.filter(
        col("linkrole").contains("StatementOfFinancialPosition") |
        col("linkrole").contains("StatementOfIncome") |
        col("linkrole").contains("http://fasb.org/us-gaap/role/disclosure/RevenuefromContractswithCustomers") ##added specifically to get a tag for Revenue that seemed fairly common
    )
    presentation_labeled = balance_sheet_df \
      .withColumn("gaap_version", lit(gaap_version_processed))\
    .join(labels_df.withColumnRenamed("concept_qname", "child"),
          on="child",
          how="left") \
    .withColumnRenamed("label_text", "child_label") \
    .join(labels_df.withColumnRenamed("concept_qname", "parent"),
          on="parent",
          how="left") \
    .withColumnRenamed("label_text", "parent_label")\

    if final_df is None:
        final_df = presentation_labeled 
    else:
        final_df = final_df.unionByName(presentation_labeled)
    
final_df.write.mode("overwrite").saveAsTable(f"{target_catalog}.{schema}.{table_name}")
